# Phase 3 findings — independent verification

This notebook recomputes every number in `docs/planning/phase3-scoring-findings.md`
directly from `data/derived/derived_candidates.csv`, before any of it gets implemented
in SQL. Each section quotes one claim from the report, recomputes it from the data, and
marks the result MATCH or MISMATCH.

I used Claude Code(or you can use any LLM thats available in your notebook i.e - Gemini in GColab, Github Copilot, etc) to help write the checks below. The prompt that generates a notebook
like this one is logged in `docs/ai-prompts-log.md` (3.2); each section below also has
its own smaller starter prompt if you just want to adapt one check.

Concept notes are generated by AI for terms that come up along the way — percentile, skew,
correlation, normalization.

### Note
The starter prompt will not generate the perfect code for the cell, it is meant to help you get started(hence the name ;) ) and you can modify based on what you are trying to verify. 


In [3]:
import pandas as pd
import numpy as np
from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

df = pd.read_csv("../data/derived/derived_candidates.csv")

# every verification check below appends one row here; the final cell turns
# this into the claim | report | notebook | match summary table
results = []

def check(claim, report_value, notebook_value, match):
    results.append({
        "claim": claim,
        "report_value": report_value,
        "notebook_value": notebook_value,
        "match": match,
    })
    print(f"{'MATCH' if match else 'MISMATCH'}: {claim}")
    print(f"  report:   {report_value}")
    print(f"  notebook: {notebook_value}")

print(df.shape)
df.head()

(1268, 12)


,cip2020_code,cip2020_title,latest_year,completions_latest_year,trend_earliest_year,completions_trend_pct,matched_soc_codes,n_socs_used,employment_weighted_openings,employment_weighted_growth_pct,in_bls_top30_flag,already_offered_by_vanderbilt
0,1.0000,"Agriculture, General",2024,360.0,2012,13.9,"19-1013, 19-4012, 19-1012",3,1.97,5.33,f,f
1,1.0101,"Agricultural Business and Management, General",2024,97.0,2012,61.7,"11-9013, 45-1011, 25-1041",3,78.99,-0.96,f,f
2,1.0102,Agribusiness/Agricultural Business Operations,2024,35.0,2012,16.7,"11-9013, 25-1041",2,84.43,-1.23,f,f
3,1.0103,Agricultural Economics,2024,184.0,2012,-57.7,"19-3011, 25-1063, 25-1041",3,0.98,2.23,f,f
4,1.0104,Farm/Farm and Ranch Management,2024,3.0,2012,-62.5,"11-9013, 45-1011, 25-9021",3,78.84,-1.04,f,f


## Verification check 1 — row counts

The report says the mart has 1,268 rows, with 14 flagged
`already_offered_by_vanderbilt = TRUE` — leaving 1,254 as the actual pool being scored.
Verifying these counts confirms the pool size before anything downstream depends on it.

**Starter prompt, if you want to adapt this check:**
> In `{DATA_FILE}`, report the total row count and the value counts for
> `{FLAG_COLUMN}`. Compare both against the source report's stated numbers and print
> MATCH or MISMATCH.

In [4]:
total_rows = len(df)

# the flag/boolean columns come in from Postgres as the strings 't'/'f',
# not Python True/False -- confirmed with df.dtypes before writing this check
vu_counts = df["already_offered_by_vanderbilt"].value_counts()
vu_true = int(vu_counts.get("t", 0))
vu_false = int(vu_counts.get("f", 0))

check(
    "total candidate rows",
    1268,
    total_rows,
    total_rows == 1268,
)
check(
    "already_offered_by_vanderbilt TRUE / FALSE",
    "14 TRUE / 1254 FALSE",
    f"{vu_true} TRUE / {vu_false} FALSE",
    vu_true == 14 and vu_false == 1254,
)

MATCH: total candidate rows
  report:   1268
  notebook: 1268
MATCH: already_offered_by_vanderbilt TRUE / FALSE
  report:   14 TRUE / 1254 FALSE
  notebook: 14 TRUE / 1254 FALSE


### Concept note: percentile and skew

A **percentile** answers "what value is X% of the way up the sorted list?" The p90 of
`completions_latest_year` is the value where 90% of programs have fewer completions than
that, and 10% have more. Median is just a nickname for the 50th percentile. Percentiles
are useful because they describe a distribution's *shape* without being thrown off by
one extreme value the way an average can be.

**Skew** measures how lopsided a distribution is — whether it's symmetric around its
center or dragged out to one side by a long tail. A skew of 0 is perfectly symmetric
(like a bell curve); the bigger the positive number, the more the distribution is
stretched toward high values by a small number of very large outliers. There isn't one
single universal formula for skew, though — pandas' `.skew()` uses a slightly different
calculation (the "adjusted Fisher-Pearson" version) than some other tools. That's worth
flagging up front: if my recomputed skew doesn't match the report's number closely, it
might just mean a different formula was used, not that either number is wrong.

## Verification check 2 — distribution profile

The report gives min, percentiles (p10–p99), max, and skew for each of the four
continuous metrics. Recomputing these directly from the data confirms the profile
before it's used to justify a normalization method.

**Report's Section 1 table:**

| Metric | n | min | p10 | p25 | median | p75 | p90 | p95 | p99 | max | skew |
|---|---|---|---|---|---|---|---|---|---|---|---|
| `completions_latest_year` | 1,268 | 0 | 0.7 | 10 | 64.5 | 326.3 | 1,227.5 | 3,309.8 | 12,572.0 | 97,150 | 17.5 |
| `completions_trend_pct` | 1,142 | -100 | -77.4 | -40.9 | 12.2 | 121.5 | 445.5 | 966.5 | 5,818.2 | 33,000 | 13.9 |
| `employment_weighted_openings` | 1,191 | 0.1 | 2.8 | 7.1 | 16.9 | 61.1 | 99.0 | 113.6 | 228.4 | 305.2 | 2.0 |
| `employment_weighted_growth_pct` | 1,191 | -25.9 | -1.2 | 1.4 | 3.7 | 6.4 | 14.2 | 16.6 | 21.4 | 26.9 | 0.8 |

**Starter prompt, if you want to adapt this check:**
> For each column in `{METRIC_COLUMNS}` in `{DATA_FILE}`, compute non-null count, min,
> max, and the p10/p25/median/p75/p90/p95/p99 percentiles, plus skew. Compare every
> value against the source report's table (allow a small tolerance for rounding and for
> skew-formula differences) and print MATCH or MISMATCH per column.

In [5]:
# report's Section 1 table, transcribed as-is for comparison
report_stats = {
    "completions_latest_year": {
        "n": 1268, "min": 0, "p10": 0.7, "p25": 10, "median": 64.5, "p75": 326.3,
        "p90": 1227.5, "p95": 3309.8, "p99": 12572.0, "max": 97150, "skew": 17.5,
    },
    "completions_trend_pct": {
        "n": 1142, "min": -100, "p10": -77.4, "p25": -40.9, "median": 12.2, "p75": 121.5,
        "p90": 445.5, "p95": 966.5, "p99": 5818.2, "max": 33000, "skew": 13.9,
    },
    "employment_weighted_openings": {
        "n": 1191, "min": 0.1, "p10": 2.8, "p25": 7.1, "median": 16.9, "p75": 61.1,
        "p90": 99.0, "p95": 113.6, "p99": 228.4, "max": 305.2, "skew": 2.0,
    },
    "employment_weighted_growth_pct": {
        "n": 1191, "min": -25.9, "p10": -1.2, "p25": 1.4, "median": 3.7, "p75": 6.4,
        "p90": 14.2, "p95": 16.6, "p99": 21.4, "max": 26.9, "skew": 0.8,
    },
}

quantile_map = {"p10": 0.10, "p25": 0.25, "median": 0.50, "p75": 0.75, "p90": 0.90, "p95": 0.95, "p99": 0.99}

def close_enough(a, b, tol_abs, tol_rel):
    # skew formulas vary slightly by library, so it gets a looser relative tolerance
    return abs(a - b) <= max(tol_abs, tol_rel * abs(b))

comparison_rows = []
for metric, report_row in report_stats.items():
    s = df[metric].dropna()
    computed = {"n": int(s.count()), "min": s.min(), "max": s.max(), "skew": s.skew()}
    for label, q in quantile_map.items():
        computed[label] = s.quantile(q)

    for stat, report_val in report_row.items():
        if stat == "n":
            tol_abs, tol_rel = 0, 0
        elif stat == "skew":
            tol_abs, tol_rel = 0.3, 0.05
        else:
            tol_abs, tol_rel = 0.3, 0.02
        comparison_rows.append({
            "metric": metric,
            "stat": stat,
            "report": report_val,
            "notebook": round(computed[stat], 2),
            "match": close_enough(computed[stat], report_val, tol_abs, tol_rel),
        })

comparison = pd.DataFrame(comparison_rows)

for metric in report_stats:
    metric_rows = comparison[comparison["metric"] == metric]
    check(
        f"distribution profile — {metric}",
        report_stats[metric],
        dict(zip(metric_rows["stat"], metric_rows["notebook"])),
        metric_rows["match"].all(),
    )

comparison

MATCH: distribution profile — completions_latest_year
  report:   {'n': 1268, 'min': 0, 'p10': 0.7, 'p25': 10, 'median': 64.5, 'p75': 326.3, 'p90': 1227.5, 'p95': 3309.8, 'p99': 12572.0, 'max': 97150, 'skew': 17.5}
  notebook: {'n': 1268.0, 'min': 0.0, 'p10': 0.7, 'p25': 10.0, 'median': 64.5, 'p75': 326.25, 'p90': 1227.5, 'p95': 3309.75, 'p99': 12571.98, 'max': 97150.0, 'skew': 17.46}
MATCH: distribution profile — completions_trend_pct
  report:   {'n': 1142, 'min': -100, 'p10': -77.4, 'p25': -40.9, 'median': 12.2, 'p75': 121.5, 'p90': 445.5, 'p95': 966.5, 'p99': 5818.2, 'max': 33000, 'skew': 13.9}
  notebook: {'n': 1142.0, 'min': -100.0, 'p10': -77.35, 'p25': -40.85, 'median': 12.2, 'p75': 121.48, 'p90': 445.45, 'p95': 966.55, 'p99': 5818.23, 'max': 33000.0, 'skew': 13.92}
MATCH: distribution profile — employment_weighted_openings
  report:   {'n': 1191, 'min': 0.1, 'p10': 2.8, 'p25': 7.1, 'median': 16.9, 'p75': 61.1, 'p90': 99.0, 'p95': 113.6, 'p99': 228.4, 'max': 305.2, 'skew': 2.0}

,metric,stat,report,notebook,match
0,completions_latest_year,n,1268.0,1268.00,True
1,completions_latest_year,min,0.0,0.00,True
2,completions_latest_year,p10,0.7,0.70,True
3,completions_latest_year,p25,10.0,10.00,True
4,completions_latest_year,median,64.5,64.50,True
5,completions_latest_year,p75,326.3,326.25,True
6,completions_latest_year,p90,1227.5,1227.50,True
7,completions_latest_year,p95,3309.8,3309.75,True
8,completions_latest_year,p99,12572.0,12571.98,True
9,completions_latest_year,max,97150.0,97150.00,True


## Verification check 3 — categorical breakdown

The report says `in_bls_top30_flag` splits 363 TRUE / 905 FALSE (28.6% true).
Verifying this confirms the flag's actual distribution before it's used as a scoring
component.

**Starter prompt, if you want to adapt this check:**
> In `{DATA_FILE}`, report the value counts (and percentage) for `{FLAG_COLUMN}`.
> Compare against the source report's stated numbers and print MATCH or MISMATCH.

In [6]:
bls_counts = df["in_bls_top30_flag"].value_counts()
bls_true = int(bls_counts.get("t", 0))
bls_false = int(bls_counts.get("f", 0))
bls_pct_true = round(100 * bls_true / (bls_true + bls_false), 1)

check(
    "in_bls_top30_flag TRUE / FALSE (% true)",
    "363 TRUE / 905 FALSE (28.6%)",
    f"{bls_true} TRUE / {bls_false} FALSE ({bls_pct_true}%)",
    bls_true == 363 and bls_false == 905,
)

MATCH: in_bls_top30_flag TRUE / FALSE (% true)
  report:   363 TRUE / 905 FALSE (28.6%)
  notebook: 363 TRUE / 905 FALSE (28.6%)


## Verification check 4 — what's driving the skew

The report says Business Administration and Management (CIP 52.0201) is the single
biggest outlier — 97,150 completions, almost 3x the #2 program, Social Work, at
34,458 — and that 127 candidates show 0 completions in their latest year, 78 of which
also show an exact -100% trend. Verifying these specific rows and counts against the
raw data confirms whether this part of the report holds up before it factors into
scoring.

**Starter prompt, if you want to adapt this check:**
> In `{DATA_FILE}`, find the top 2 rows by `{METRIC_COLUMN}` and report their
> identifying columns and values. Separately, count rows where `{METRIC_COLUMN} == 0`,
> and within that subset, count how many also have `{TREND_COLUMN}` exactly equal to
> `{TREND_VALUE}`. Compare every count against the source report's stated numbers and
> print MATCH or MISMATCH for each.

In [7]:
top2 = df.nlargest(2, "completions_latest_year")[["cip2020_code", "cip2020_title", "completions_latest_year"]]
display(top2)

top1_code, top1_completions = top2.iloc[0]["cip2020_code"], top2.iloc[0]["completions_latest_year"]
top2_code, top2_completions = top2.iloc[1]["cip2020_code"], top2.iloc[1]["completions_latest_year"]

check(
    "#1 by completions is CIP 52.0201 with 97,150",
    "52.0201, 97150",
    f"{top1_code}, {top1_completions}",
    top1_code == 52.0201 and top1_completions == 97150,
)
check(
    "#2 by completions is CIP 44.0701 with 34,458",
    "44.0701, 34458",
    f"{top2_code}, {top2_completions}",
    top2_code == 44.0701 and top2_completions == 34458,
)

zero_mask = df["completions_latest_year"] == 0
n_zero = int(zero_mask.sum())
n_zero_2024 = int((df.loc[zero_mask, "latest_year"] == 2024).sum())
n_zero_neg100 = int((df.loc[zero_mask, "completions_trend_pct"] == -100).sum())

check("candidates with 0 completions_latest_year", 127, n_zero, n_zero == 127)
check("of those, latest_year == 2024", 87, n_zero_2024, n_zero_2024 == 87)
check("of those, trend == exactly -100%", 78, n_zero_neg100, n_zero_neg100 == 78)

,cip2020_code,cip2020_title,completions_latest_year
1174,52.0201,"Business Administration and Management, General",97150.0
876,44.0701,Social Work,34458.0


MATCH: #1 by completions is CIP 52.0201 with 97,150
  report:   52.0201, 97150
  notebook: 52.0201, 97150.0
MATCH: #2 by completions is CIP 44.0701 with 34,458
  report:   44.0701, 34458
  notebook: 44.0701, 34458.0
MATCH: candidates with 0 completions_latest_year
  report:   127
  notebook: 127
MATCH: of those, latest_year == 2024
  report:   87
  notebook: 87
MATCH: of those, trend == exactly -100%
  report:   78
  notebook: 78


### Concept note: correlation coefficient

A **correlation coefficient** (here, Pearson's r) measures how strongly two numbers move
together, on a scale from -1 to +1. +1 means "when one goes up, the other always goes up
proportionally," -1 means the opposite (perfectly inverse), and 0 means no linear
relationship at all. A value like 0.04–0.20 is close to 0: the four metrics barely move
together, which is good for scoring, since each one adds real, independent signal rather
than repeating another one's information. (If two metrics were 0.9 correlated, combining
both would mostly count the same signal twice.)

## Verification check 5 — correlation between the four continuous metrics

The report says all six pairwise correlations between the four continuous metrics fall
between 0.04 and 0.20 (pairwise complete observations only) — essentially independent.
Verifying this confirms whether combining all four metrics in a composite score adds
independent signal or risks double-counting.

**Starter prompt, if you want to adapt this check:**
> In `{DATA_FILE}`, compute the pairwise Pearson correlation matrix for
> `{METRIC_COLUMNS}` using pairwise-complete observations. Extract the six unique
> off-diagonal values and report the min and max. Compare against the source report's
> stated range and print MATCH or MISMATCH.

In [8]:
import itertools

continuous_cols = [
    "completions_latest_year",
    "completions_trend_pct",
    "employment_weighted_openings",
    "employment_weighted_growth_pct",
]

# DataFrame.corr() uses pairwise-complete observations per pair by default,
# which matches how the report says it computed this
corr_matrix = df[continuous_cols].corr()
display(corr_matrix)

pairwise_corrs = {
    f"{a} vs {b}": corr_matrix.loc[a, b]
    for a, b in itertools.combinations(continuous_cols, 2)
}
min_corr, max_corr = min(pairwise_corrs.values()), max(pairwise_corrs.values())

check(
    "all six pairwise correlations fall in [0.04, 0.20]",
    "[0.04, 0.20]",
    f"[{min_corr:.2f}, {max_corr:.2f}]",
    0.03 <= min_corr <= 0.05 and 0.19 <= max_corr <= 0.21,
)
pd.Series(pairwise_corrs, name="pearson_r").sort_values()

,completions_latest_year,completions_trend_pct,employment_weighted_openings,employment_weighted_growth_pct
completions_latest_year,1.000000,0.036162,0.197629,0.06214
completions_trend_pct,0.036162,1.000000,0.046389,0.07994
employment_weighted_openings,0.197629,0.046389,1.000000,0.07379
employment_weighted_growth_pct,0.062140,0.079940,0.073790,1.00000


MATCH: all six pairwise correlations fall in [0.04, 0.20]
  report:   [0.04, 0.20]
  notebook: [0.04, 0.20]


completions_latest_year vs completions_trend_pct                  0.036162
completions_trend_pct vs employment_weighted_openings             0.046389
completions_latest_year vs employment_weighted_growth_pct         0.062140
employment_weighted_openings vs employment_weighted_growth_pct    0.073790
completions_trend_pct vs employment_weighted_growth_pct           0.079940
completions_latest_year vs employment_weighted_openings           0.197629
Name: pearson_r, dtype: float64

## Verification check 6 — data-quality flag (SOC 11-1021 concentration)

The report flags that the top 5 candidates by `employment_weighted_openings` —
Parks/Rec Facilities Management (31.0399), Parks/Rec Facilities Management General
(31.0301), Risk Management (52.0215), Retail Management (52.0212), Management Science
(52.1301) — all match SOC 11-1021 ("General and Operations Managers"). Verifying this
confirms the data-quality flag before using the openings ranking as-is.

**Starter prompt, if you want to adapt this check:**
> In `{DATA_FILE}`, find the top 5 rows by `{METRIC_COLUMN}` and check whether
> `{SOC_CODE}` appears in each row's `{SOC_COLUMN}`. Compare the resulting CIP codes
> and match/no-match result against the source report's stated findings and print
> MATCH or MISMATCH.

In [7]:
top5_openings = df.nlargest(5, "employment_weighted_openings")[
    ["cip2020_code", "cip2020_title", "employment_weighted_openings", "matched_soc_codes"]
]
display(top5_openings)

top5_codes = set(top5_openings["cip2020_code"])
report_codes = {31.0399, 31.0301, 52.0215, 52.0212, 52.1301}
all_have_soc = top5_openings["matched_soc_codes"].str.contains("11-1021").all()

check(
    "top 5 by employment_weighted_openings are these 5 CIPs",
    sorted(report_codes),
    sorted(top5_codes),
    top5_codes == report_codes,
)
check(
    "all 5 include SOC 11-1021 in matched_soc_codes",
    True,
    bool(all_have_soc),
    all_have_soc,
)

,cip2020_code,cip2020_title,employment_weighted_openings,matched_soc_codes
712,31.0399,"Parks, Recreation, and Leisure Facilities Mana...",305.21,"11-1021, 11-9072"
710,31.0301,"Parks, Recreation, and Leisure Facilities Mana...",293.90,"11-1021, 11-3013, 11-9072"
1187,52.0215,Risk Management,257.41,"11-1021, 11-3031, 13-2053"
1185,52.0212,Retail Management,252.68,"11-1021, 41-1011, 25-1011"
1237,52.1301,Management Science,246.89,"11-1021, 13-1111, 11-1011"


MATCH: top 5 by employment_weighted_openings are these 5 CIPs
  report:   [31.0301, 31.0399, 52.0212, 52.0215, 52.1301]
  notebook: [31.0301, 31.0399, 52.0212, 52.0215, 52.1301]
MATCH: all 5 include SOC 11-1021 in matched_soc_codes
  report:   True
  notebook: True


## Verification check 7 — NULL overlap

The report says two NULL patterns exist independently — 77 candidates with no
labor-market match, 126 with no completions trend — with 11 overlapping, leaving 192
total incomplete and 1,076 fully complete. Checking this overlap directly confirms
whether the report's numbers are correct before they inform how missing data gets
handled in scoring.

**Starter prompt, if you want to adapt this check:**
> In `{DATA_FILE}`, count NULLs in `{COLUMN_A}` and `{COLUMN_B}` separately, then count
> rows where both are NULL, where at least one is NULL, and where neither is NULL.
> Compare each count (and its share of total rows) against the source report's stated
> numbers and print MATCH or MISMATCH.

In [10]:
no_labor_match = df["employment_weighted_openings"].isna()
no_trend = df["completions_trend_pct"].isna()
both_missing = no_labor_match & no_trend
any_missing = no_labor_match | no_trend
fully_complete = ~any_missing

n_no_labor = int(no_labor_match.sum())
n_no_trend = int(no_trend.sum())
n_both = int(both_missing.sum())
n_any = int(any_missing.sum())
n_complete = int(fully_complete.sum())
pct_any = round(100 * n_any / len(df), 1)
pct_complete = round(100 * n_complete / len(df), 1)

check("no-labor-match (employment_weighted_openings NULL)", 77, n_no_labor, n_no_labor == 77)
check("no-trend (completions_trend_pct NULL)", 126, n_no_trend, n_no_trend == 126)
check("candidates missing both", 11, n_both, n_both == 11)
check("missing at least one metric (total, %)", "192 (15.1%)", f"{n_any} ({pct_any}%)", n_any == 192)
check("fully complete (total, %)", "1076 (84.9%)", f"{n_complete} ({pct_complete}%)", n_complete == 1076)

MATCH: no-labor-match (employment_weighted_openings NULL)
  report:   77
  notebook: 77
MATCH: no-trend (completions_trend_pct NULL)
  report:   126
  notebook: 126
MATCH: candidates missing both
  report:   11
  notebook: 11
MATCH: missing at least one metric (total, %)
  report:   192 (15.1%)
  notebook: 192 (15.1%)
MATCH: fully complete (total, %)
  report:   1076 (84.9%)
  notebook: 1076 (84.9%)


### Concept note: min-max scaling vs. percentile-rank scaling

The four continuous metrics live on completely different scales (completions run into
the tens of thousands; growth is a percentage), so they can't be averaged directly —
they first need to be squeezed onto one common 0–100 scale. Two ways to do that:

- **Min-max scaling**: `(value - min) / (max - min) × 100`. Whatever the single smallest
  observed value is becomes 0, whatever the single largest is becomes 100, and
  everything else is stretched proportionally in between. Simple, but it means *one*
  extreme outlier single-handedly defines what "100" means for everyone else.
- **Percentile-rank scaling**: replace each value with its percentile position in the
  distribution (from the concept note above) — "you're higher than 91% of programs"
  becomes a score of 91, regardless of *how much* higher. It only cares about order, not
  magnitude, so one giant outlier can't compress everyone else.

The report recommends percentile rank specifically because `completions_latest_year` is
so skewed (skew ≈ 17.5, verified above), and claims min-max scaling would let one
outlier (Business Administration, 97,150 completions) crush almost every other program
down near a score of 0. The next cell computes real scores both ways for three actual
rows to check that claim directly.

## Verification check 8 — empirical normalization tradeoff

The report claims min-max scaling would crush the median candidate's score near zero
because of Business Administration's outlier, and recommends percentile-rank scaling
instead. Computing both scores on real rows confirms whether that tradeoff is actually
as described before adopting percentile rank as the normalization method.

**Starter prompt, if you want to adapt this check:**
> In `{DATA_FILE}`, pick three rows from `{METRIC_COLUMN}`: the row closest to the
> median, a row near a high percentile (e.g. p95), and the row with the maximum value.
> Compute each row's score two ways — min-max `(x-min)/(max-min)*100` and
> percentile-rank `rank(pct=True)*100` — and report both side by side to show how the
> two methods diverge.

In [11]:
metric = "completions_latest_year"
s = df[metric]
min_val, max_val = s.min(), s.max()

# real rows: closest-to-median, closest-to-p95 ("near the top"), and the true max (outlier)
median_idx = (s - s.median()).abs().idxmin()
near_top_idx = (s - s.quantile(0.95)).abs().idxmin()
outlier_idx = s.idxmax()

percentile_rank = s.rank(pct=True) * 100

demo_rows = []
for label, idx in [("median row", median_idx), ("near-top row (~p95)", near_top_idx), ("extreme outlier", outlier_idx)]:
    value = s[idx]
    minmax_score = (value - min_val) / (max_val - min_val) * 100
    pctrank_score = percentile_rank[idx]
    demo_rows.append({
        "row": label,
        "cip_title": df.loc[idx, "cip2020_title"],
        "completions_latest_year": value,
        "min_max_score": round(minmax_score, 2),
        "percentile_rank_score": round(pctrank_score, 2),
    })

demo_df = pd.DataFrame(demo_rows)
display(demo_df)

median_minmax_score = demo_df.loc[demo_df["row"] == "median row", "min_max_score"].iloc[0]
check(
    "median candidate's min-max score is ≈0.07/100",
    "≈0.07",
    median_minmax_score,
    abs(median_minmax_score - 0.07) < 0.1,
)

,row,cip_title,completions_latest_year,min_max_score,percentile_rank_score
0,median row,"Plant Sciences, Other",64.0,0.07,49.80
1,near-top row (~p95),Computer/Information Technology Services Admin...,3329.0,3.43,95.03
2,extreme outlier,"Business Administration and Management, General",97150.0,100.00,100.00


MATCH: median candidate's min-max score is ≈0.07/100
  report:   ≈0.07
  notebook: 0.07


This matches what the report predicted. The median row scores about 0.07/100 under
min-max, but about 50/100 under percentile rank. That's the difference between looking
like the worst program in the pool and looking like an average one. The near-top row
makes it even clearer: 3,329 completions puts it in the 95th percentile, but min-max
still only gives it a 3.4/100, because Business Administration's outlier sets the
ceiling for everyone else.

## Summary

Every `check()` call above appended one row to `results`, so the table below is built
straight from that list instead of retyped by hand.

In [10]:
summary = pd.DataFrame(results)
summary["match"] = summary["match"].map({True: "yes", False: "NO"})

mismatches = summary[summary["match"] == "NO"]
if len(mismatches):
    print(f"MISMATCHES FOUND ({len(mismatches)}):")
    display(mismatches)
else:
    print("No mismatches found — every recomputed value matched the report within tolerance.")

pd.set_option("display.max_colwidth", None)
summary

No mismatches found — every recomputed value matched the report within tolerance.


,claim,report_value,notebook_value,match
0,total candidate rows,1268,1268,yes
1,already_offered_by_vanderbilt TRUE / FALSE,14 TRUE / 1254 FALSE,14 TRUE / 1254 FALSE,yes
2,distribution profile — completions_latest_year,"{'n': 1268, 'min': 0, 'p10': 0.7, 'p25': 10, 'median': 64.5, 'p75': 326.3, 'p90': 1227.5, 'p95': 3309.8, 'p99': 12572.0, 'max': 97150, 'skew': 17.5}","{'n': 1268.0, 'min': 0.0, 'p10': 0.7, 'p25': 10.0, 'median': 64.5, 'p75': 326.25, 'p90': 1227.5, 'p95': 3309.75, 'p99': 12571.98, 'max': 97150.0, 'skew': 17.46}",yes
3,distribution profile — completions_trend_pct,"{'n': 1142, 'min': -100, 'p10': -77.4, 'p25': -40.9, 'median': 12.2, 'p75': 121.5, 'p90': 445.5, 'p95': 966.5, 'p99': 5818.2, 'max': 33000, 'skew': 13.9}","{'n': 1142.0, 'min': -100.0, 'p10': -77.35, 'p25': -40.85, 'median': 12.2, 'p75': 121.48, 'p90': 445.45, 'p95': 966.55, 'p99': 5818.23, 'max': 33000.0, 'skew': 13.92}",yes
4,distribution profile — employment_weighted_openings,"{'n': 1191, 'min': 0.1, 'p10': 2.8, 'p25': 7.1, 'median': 16.9, 'p75': 61.1, 'p90': 99.0, 'p95': 113.6, 'p99': 228.4, 'max': 305.2, 'skew': 2.0}","{'n': 1191.0, 'min': 0.1, 'p10': 2.8, 'p25': 7.1, 'median': 16.87, 'p75': 61.05, 'p90': 99.02, 'p95': 113.64, 'p99': 228.37, 'max': 305.21, 'skew': 2.03}",yes
5,distribution profile — employment_weighted_growth_pct,"{'n': 1191, 'min': -25.9, 'p10': -1.2, 'p25': 1.4, 'median': 3.7, 'p75': 6.4, 'p90': 14.2, 'p95': 16.6, 'p99': 21.4, 'max': 26.9, 'skew': 0.8}","{'n': 1191.0, 'min': -25.9, 'p10': -1.16, 'p25': 1.42, 'median': 3.67, 'p75': 6.37, 'p90': 14.17, 'p95': 16.6, 'p99': 21.41, 'max': 26.91, 'skew': 0.81}",yes
6,in_bls_top30_flag TRUE / FALSE (% true),363 TRUE / 905 FALSE (28.6%),363 TRUE / 905 FALSE (28.6%),yes
7,"#1 by completions is CIP 52.0201 with 97,150","52.0201, 97150","52.0201, 97150.0",yes
8,"#2 by completions is CIP 44.0701 with 34,458","44.0701, 34458","44.0701, 34458.0",yes
9,candidates with 0 completions_latest_year,127,127,yes


# Conclusion
All 21 checks came back a match. The report's numbers hold up against the real data.

Next I'll work on the normalization method to see what to use, weight the five components, and handle the rows with missing metrics.

A few of the top programs by employment_weighted_openings rank high largely because they share the same broad "General and Operations Managers" match, so I'd treat that specific ranking with some skepticism no matter which weighting method I pick.

Once the method is figured out, I can build the score in SQL, then set the Go/Test/Pass cutoffs from whatever the real score distribution turns out to be.